In [ ]:
from torch.utils.data import DataLoader
from torch import nn
from torch import optim
import torch


device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from torchvision.transforms import v2
from medmnist import PathMNIST

tf = v2.Compose([
    v2.ToTensor(),
])

train_dataset = PathMNIST(root="./data/",split="train",transform=tf,download=True,size=64)
val_dataset = PathMNIST(root="./data/",split="val",transform=tf,download=True,size=64)

n_labels = len(train_dataset.info["label"].items())

/mnt/sdc/Hasan/Documents/My Documents/Code/MAE-Model-MedMNIST-Predictor/.venv/lib64/python3.14/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [ ]:
from 
model = Predictor(n_labels=n_labels)
model.load_state_dict(torch.load("model_weights/v3_checkpoints/model_epoch_50.pt"))

print(model)

Predictor(
  (autoencoder): AutoEncoder(
    (patcher): CNN(
      (conv): Conv2d(3, 1024, kernel_size=(4, 4), stride=(4, 4))
      (fc): Linear(in_features=1024, out_features=512, bias=True)
    )
    (encoder): Encoder(
      (transformer_blocks): Sequential(
        (0): TransformerBlock(
          (mha): MultiHeadAttention(
            (linear_qkv): Linear(in_features=512, out_features=1536, bias=True)
            (linear_out): Linear(in_features=512, out_features=512, bias=True)
          )
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (ff): FeedForward(
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
            (gelu): GELU(approximate='none')
          )
        )
        (1): TransformerBlock(
          (mha): MultiHeadAttention(
            (linear_q

In [5]:

import os
num_workers = max(1, os.cpu_count() - 2) 

train_batch_size = 64
val_batch_size = 64
effective_batch = 256
if 256 % train_batch_size != 0:
    raise ValueError()

train_dl = DataLoader(
    train_dataset,
    batch_size= train_batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)

val_dl = DataLoader(
    val_dataset,
    batch_size = val_batch_size,
    num_workers=num_workers,
    pin_memory=True,
)


In [ ]:
import math

grad_acc = max(1,(256//train_batch_size))
steps_per_epoch = math.ceil(len(train_dl) / grad_acc)

epochs = 100
warm_up_epochs = 10
checkpointing_rate = 5

warm_up_steps = warm_up_epochs*steps_per_epoch
cosine_steps = (epochs - warm_up_epochs)*steps_per_epoch

base_lr = 1.5e-4
max_lr = base_lr * (effective_batch/256)
min_lr = 1e-6

loss = nn.CrossEntropyLoss()
optimiser = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),max_lr)
# start at minimum and go up
warmup_scheduler = optim.lr_scheduler.LinearLR(
    optimiser, start_factor=1e-2, end_factor=1.0, total_iters=warm_up_steps
)
cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimiser, T_max=cosine_steps, eta_min=min_lr
)
scheduler = optim.lr_scheduler.SequentialLR(
    optimiser, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warm_up_steps]
)


In [7]:
from training_functions import train,test
from tqdm.notebook import tqdm
import time
import csv

log_file_path = f"logs/v3_finetune_log_{time.time()}.csv"


with open(log_file_path, mode="w", newline="") as f:

    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss"])

    model = model.to(device)
    torch.set_float32_matmul_precision('high')
    model = torch.compile(model)
    with tqdm(range(epochs),desc="Epochs") as bar:
        print("_______________________________________________")
        for epoch in bar:
            train_loss = train(model, device, train_dl, loss, optimiser, epoch, scheduler, grad_acc)
            torch.cuda.empty_cache()
            val_loss = test(model, device, val_dl, loss,epoch)
            torch.cuda.empty_cache()
            print("_______________________________________________")
            bar.set_postfix({"current val_loss": f"{val_loss:.4f}"})
            writer.writerow([epoch+1, train_loss, val_loss])
            f.flush()

            if (epoch+1) % checkpointing_rate == 0:
                torch.save(model._orig_mod.state_dict() if hasattr(model, '_orig_mod') else model.state_dict(),f"model_weights/v3_finetune_checkpoints/model_epoch_{epoch+1}.pt")

Epochs:   0%|          | 0/100 [00:00<?, ?it/s]

_______________________________________________


Epoch 1: Training:   0%|          | 0/1407 [00:00<?, ?it/s]

W0730 19:39:16.383000 1275228 torch/_inductor/utils.py:1953] [0/0] Not enough SMs to use max_autotune_gemm mode


Average Train CE Loss: 0.032575


...

Average Train CE Loss: 0.009127


Epoch 18: Validation:   0%|          | 0/157 [00:00<?, ?it/s]

Average Test CE Loss: 0.060454
_______________________________________________


Epoch 19: Training:   0%|          | 0/1407 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import time

dataframe = pd.read_csv(log_file_path)

plt.plot(dataframe["epoch"],dataframe["train_loss"])
plt.plot(dataframe["epoch"],dataframe["val_loss"])
plt.title
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.savefig(f"graphs/finetune/finetune_v3_loss_{time.time()}.png")
plt.show()
plt.close()